In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

In [2]:
# load data
df = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
df_test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

## EDA

In [3]:
df.head()
df.tail()
df.info()
df.isna().sum()
df.nunique()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False
8692,9280_02,Europa,False,E/608/S,TRAPPIST-1e,44.0,False,126.0,4688.0,0.0,0.0,12.0,Propsh Hontichre,True


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

PassengerId     8693
HomePlanet         3
CryoSleep          2
Cabin           6560
Destination        3
Age               80
VIP                2
RoomService     1273
FoodCourt       1507
ShoppingMall    1115
Spa             1327
VRDeck          1306
Name            8473
Transported        2
dtype: int64

## Data Preprocessing / Feature Engineering

In [4]:
cat_col = ['CryoSleep', 'HomePlanet', 'Destination', 'VIP']
num_col = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

def feature_engineering(df):
    df[['Group', 'NumberInGroup']] = df['PassengerId'].str.split('_', expand=True)
    df[['Deck', 'CabinNumber', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df[['FirstName', 'LastName']] = df['Name'].str.split(' ', expand=True)
    
    df['TotalExpenses'] = df[num_col].sum(axis=1)

    # anyone w/ 0 as their total expenses will be assumed to be in CryoSleep
    df.loc[df['TotalExpenses'] == 0, 'CryoSleep'] = True
    
    for col in num_col:
        df[col] = df[col].fillna(0)
    
    for col in cat_col:
        df[col] = df[col].fillna(df[col].mode()[0])
    
    df['Age'] = df['Age'].fillna(df['Age'].median())

    drop_col = ['Name', 'Cabin']
    
    df = df.drop(columns=drop_col)
    
    cabin_details = ['Deck', 'CabinNumber', 'Side']
    full_name = ['FirstName', 'LastName']
    
    # replace CabinNumber missing values w/ -9999 (int)
    for col in [cabin_details[1]]:
        df[col] = df[col].fillna(-9999)
        
    # replace Deck and Side missing values w/ "N/A" (str)
    for col in [cabin_details[0], cabin_details[2]]:
        df[col] = df[col].fillna("N/A")
        
    for col in full_name:
        df[col] = df[col].fillna(-8888)

    df['FamilyId'] = df['Group'].astype(str) + '_' + df['LastName'].astype(str)
    
    # counts how many people w/ same family id
    family_counts = df.groupby('FamilyId').size()
    
    df['FamilySize'] = df['FamilyId'].map(family_counts)
    df['IsFamily'] = (df['FamilySize'] > 1).astype(int)
    
    # dropping columns that will cause overfitting later on
    final_drop=['FamilyId', 'PassengerId', 'FirstName', 'LastName']
    
    df.drop(columns=final_drop, inplace=True)
    return df
    

In [5]:
train = feature_engineering(df)
test = feature_engineering(df_test)

train.head()
test.head()

/tmp/ipykernel_16/3840675285.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])
/tmp/ipykernel_16/3840675285.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Group,NumberInGroup,Deck,CabinNumber,Side,TotalExpenses,FamilySize,IsFamily
0,Europa,True,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False,0001,01,B,0,P,0.0,1,0
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True,0002,01,F,0,S,736.0,1,0
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,0003,01,A,0,S,10383.0,2,1
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,0003,02,A,0,S,5176.0,2,1
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True,0004,01,F,1,S,1091.0,1,0


,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Group,NumberInGroup,Deck,CabinNumber,Side,TotalExpenses,FamilySize,IsFamily
0,Earth,True,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,0013,01,G,3,S,0.0,1,0
1,Earth,False,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,0018,01,F,4,S,2832.0,1,0
2,Europa,True,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,0019,01,C,0,S,0.0,1,0
3,Europa,False,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,0021,01,C,1,S,7418.0,1,0
4,Earth,False,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,0023,01,F,5,S,645.0,1,0


In [6]:
le = LabelEncoder()
need_encoding = ['HomePlanet', 'Destination', 'Deck', 'Side'] # deck + side have multiple dtypes
int_convert = ['CryoSleep', 'VIP', 'Transported', 'Group', 'NumberInGroup', 'CabinNumber']
int_convert_test = ['CryoSleep', 'VIP', 'Group', 'NumberInGroup', 'CabinNumber']

for col in int_convert:
    train[col] = train[col].astype(int)

for col in int_convert_test:
    test[col] = test[col].astype(int)

for col in need_encoding: 
    train[f'{col}Encoded'] = le.fit_transform(train[col])
    test[f'{col}Encoded'] = le.fit_transform(test[col])

train = train.drop(columns=need_encoding)
test = test.drop(columns=need_encoding)

train.head()
test.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Group,NumberInGroup,CabinNumber,TotalExpenses,FamilySize,IsFamily,HomePlanetEncoded,DestinationEncoded,DeckEncoded,SideEncoded
0,1,39.0,0,0.0,0.0,0.0,0.0,0.0,0,1,1,0,0.0,1,0,1,2,1,1
1,0,24.0,0,109.0,9.0,25.0,549.0,44.0,1,2,1,0,736.0,1,0,0,2,5,2
2,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0,0,3,1,0,10383.0,2,1,1,2,0,2
3,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0,0,3,2,0,5176.0,2,1,1,2,0,2
4,0,16.0,0,303.0,70.0,151.0,565.0,2.0,1,4,1,1,1091.0,1,0,0,2,5,2


,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Group,NumberInGroup,CabinNumber,TotalExpenses,FamilySize,IsFamily,HomePlanetEncoded,DestinationEncoded,DeckEncoded,SideEncoded
0,1,27.0,0,0.0,0.0,0.0,0.0,0.0,13,1,3,0.0,1,0,0,2,6,2
1,0,19.0,0,0.0,9.0,0.0,2823.0,0.0,18,1,4,2832.0,1,0,0,2,5,2
2,1,31.0,0,0.0,0.0,0.0,0.0,0.0,19,1,0,0.0,1,0,1,0,2,2
3,0,38.0,0,0.0,6652.0,0.0,181.0,585.0,21,1,1,7418.0,1,0,1,2,2,2
4,0,20.0,0,10.0,0.0,635.0,0.0,0.0,23,1,5,645.0,1,0,0,2,5,2


In [7]:
X = train.drop(columns='Transported')
y = train['Transported'] 

scaler = StandardScaler()

standard_df = scaler.fit_transform(train, y)
standard_df = pd.DataFrame(standard_df, columns=train.columns)

X = standard_df.drop(columns='Transported')
y = train['Transported'] 

X.head()
y.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Group,NumberInGroup,CabinNumber,TotalExpenses,FamilySize,IsFamily,HomePlanetEncoded,DestinationEncoded,DeckEncoded,SideEncoded
0,1.174601,0.711945,-0.153063,-0.333105,-0.281027,-0.283579,-0.270626,-0.263003,-1.734409,-0.491161,-0.214978,-0.514066,-0.610822,-0.811769,0.440385,0.620545,-1.866494,-0.866174
1,-0.851353,-0.334037,-0.153063,-0.168073,-0.275387,-0.241771,0.217158,-0.224205,-1.734034,-0.491161,-0.214978,-0.251479,-0.610822,-0.811769,-0.817259,0.620545,0.350474,0.975267
2,-0.851353,2.036857,6.533255,-0.268001,1.959998,-0.283579,5.695623,-0.219796,-1.733660,-0.491161,-0.214978,3.190333,0.176993,1.231877,0.440385,0.620545,-2.420736,0.975267
3,-0.851353,0.293552,-0.153063,-0.333105,0.523010,0.336851,2.687176,-0.092818,-1.733660,0.457443,-0.214978,1.332604,0.176993,1.231877,0.440385,0.620545,-2.420736,0.975267
4,-0.851353,-0.891895,-0.153063,0.125652,-0.237159,-0.031059,0.231374,-0.261240,-1.733286,-0.491161,-0.214378,-0.124824,-0.610822,-0.811769,-0.817259,0.620545,0.350474,0.975267


0    0
1    1
2    0
3    0
4    1
Name: Transported, dtype: int64

## Model Building

In [8]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# split data 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2)

# create model instance
bst = XGBClassifier()

# fit model
bst.fit(X_train, y_train)

# make predictions
preds = bst.predict(X_test)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [9]:
# KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_val_score(bst, X_test, y_test, cv=kf, scoring='accuracy')

print("Accuracy scores for each fold:", scores)
print("Mean Accuracy:", np.mean(scores))

Accuracy scores for each fold: [0.77586207 0.79310345 0.84482759 0.77011494 0.81034483 0.82183908
 0.78735632 0.79310345 0.83333333 0.76878613]
Mean Accuracy: 0.7998671184638894


In [10]:
# submission

standard_test = scaler.fit_transform(test)
X_test = pd.DataFrame(standard_test, columns=test.columns)

X_preds = bst.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': df_test['PassengerId'],
    'Transported': X_preds
})

submission.to_csv('out.csv', index=False)

submission.head()

,PassengerId,Transported
0,0013_01,0
1,0018_01,0
2,0019_01,1
3,0021_01,1
4,0023_01,1
